<a href="https://colab.research.google.com/github/nurik030608-jpg/tap.model/blob/main/ProgrammOptimizer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [21]:
import pandas as pd
import numpy as np
import re
import os
from dataclasses import dataclass
from typing import Dict, List, Tuple
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split


In [22]:
CATEGORY_FILES = {
    "cpu": "CPUData.csv",
    "gpu": "GPUData.csv",
    "motherboard": "MotherboardData.csv",
    "ram": "RAMData.csv",
    "ssd": "SSDData.csv",
    "hdd": "HDDData.csv",
    "psu": "PSUData.csv",
    "case": "CaseData.csv",
    "cpucooler": "CPUCoolerData.csv",
    "monitor": "MonitorData.csv",
}

In [23]:
def first_number(value, default=np.nan):
    if pd.isna(value):
        return default
    if isinstance(value, (int, float, np.integer, np.floating)):
        return float(value)
    text = str(value).replace(",", "")
    m = re.search(r"(-?\d+(?:\.\d+)?)", text)
    return float(m.group(1)) if m else default

def parse_price(value):
    return first_number(value, np.nan)

def parse_bool(value):
    if pd.isna(value):
        return 0
    text = str(value).strip().lower()
    return 1 if text in {"true", "yes", "1"} else 0

def parse_storage_to_gb(value):
    n = first_number(value, np.nan)
    if pd.isna(n):
        return np.nan
    text = str(value).lower()
    if "tb" in text:
        return n * 1024
    return n

def parse_resolution_pixels(value):
    if pd.isna(value):
        return np.nan
    m = re.search(r"(\d+)\s*x\s*(\d+)", str(value).lower())
    if not m:
        return np.nan
    return int(m.group(1)) * int(m.group(2))

def ram_type_family(value):
    if pd.isna(value):
        return None
    s = str(value).upper()
    m = re.search(r"(DDR\d)", s)
    return m.group(1) if m else s

def supported_socket_list(value):
    if pd.isna(value):
        return []
    return [x.strip().upper() for x in str(value).split(",")]

def case_supported_form_factors(case_value):
    if pd.isna(case_value):
        return set()
    v = str(case_value).strip()
    hierarchy = {
        "E-ATX": {"E-ATX", "ATX", "MICRO-ATX", "MINI-ITX"},
        "ATX": {"ATX", "MICRO-ATX", "MINI-ITX"},
        "MICRO-ATX": {"MICRO-ATX", "MINI-ITX"},
        "MINI-ITX": {"MINI-ITX"},
    }
    key = v.upper()
    return hierarchy.get(key, {key})

def motherboard_form_factor(value):
    if pd.isna(value):
        return None
    return str(value).strip().upper()


In [24]:
class PCBuildOptimizer:
    def __init__(self, data_dir: str):
        self.data_dir = data_dir
        self.data = {name: pd.read_csv(os.path.join(data_dir, file)) for name, file in CATEGORY_FILES.items()}
        self._preprocess_all()
        self.stage1_models = {}
        self.stage2_model = None
        self.stage3_model = None

    def _preprocess_all(self):
        for name, df in self.data.items():
            df = df.copy()
            df["price_num"] = df["Price"].apply(parse_price)
            if "Product Page" not in df.columns:
                df["Product Page"] = None

            if name == "cpu":
                df["base_clock_num"] = df["Base Clock"].apply(first_number)
                df["turbo_clock_num"] = df["Turbo Clock"].apply(first_number)
                df["cores_num"] = pd.to_numeric(df["Cores"], errors="coerce")
                df["threads_num"] = pd.to_numeric(df["Threads"], errors="coerce")
                df["tdp_num"] = df["TDP"].apply(first_number)
                df["socket_norm"] = df["Socket"].astype(str).str.upper().str.strip()
                df["igpu_flag"] = df["Integrated GPU"].apply(lambda x: 0 if str(x).strip().lower() in {"false", "nan", "none"} else 1)
            elif name == "gpu":
                df["boost_clock_num"] = df["Boost Clock"].apply(first_number)
                df["vram_num"] = df["Vram"].apply(first_number)
                df["memory_clock_num"] = df["Memory Clock"].apply(first_number)
                df["tdp_num"] = df["TDP"].apply(first_number)
                df["length_num"] = df["Length"].apply(first_number)
                df["pin8_num"] = pd.to_numeric(df["8-Pin Connectors"], errors="coerce").fillna(0)
                df["pin6_num"] = pd.to_numeric(df["6-Pin Connectors"], errors="coerce").fillna(0)
            elif name == "motherboard":
                df["socket_norm"] = df["Socket"].astype(str).str.upper().str.strip()
                df["memory_type_norm"] = df["Memory Type"].apply(ram_type_family)
                df["mem_capacity_num"] = df["Memory Capacity"].apply(parse_storage_to_gb)
                df["ram_slots_num"] = pd.to_numeric(df["RAM Slots"], errors="coerce")
                df["wifi_flag"] = df["WiFi"].apply(parse_bool)
                df["form_factor_norm"] = df["Form Factor"].apply(motherboard_form_factor)
            elif name == "ram":
                df["memory_type_norm"] = df["Ram Type"].apply(ram_type_family)
                df["size_num"] = df["Size"].apply(parse_storage_to_gb)
                df["clock_num"] = pd.to_numeric(df["Clock"], errors="coerce")
                df["sticks_num"] = pd.to_numeric(df["Sticks"], errors="coerce")
            elif name == "ssd":
                df["size_num"] = df["Size"].apply(parse_storage_to_gb)
                df["nvme_flag"] = df["Protocol"].astype(str).str.upper().str.contains("NVM", na=False).astype(int)
                df["tlc_flag"] = df["NAND"].astype(str).str.upper().str.contains("TLC", na=False).astype(int)
            elif name == "hdd":
                df["size_num"] = df["Size"].apply(parse_storage_to_gb)
                df["rpm_num"] = pd.to_numeric(df["RPM"], errors="coerce")
                df["cache_num"] = df["Cache"].apply(first_number)
            elif name == "psu":
                df["watt_num"] = df["Watt"].apply(first_number)
                df["efficiency_score"] = (
                    df["Efficiency Rating"].astype(str)
                    .str.extract(r"(Titanium|Platinum|Gold|Silver|Bronze)", expand=False)
                    .map({"Bronze": 1, "Silver": 2, "Gold": 3, "Platinum": 4, "Titanium": 5})
                    .fillna(0)
                )
                df["size_norm"] = df["Size"].astype(str).str.upper().str.strip()
            elif name == "case":
                df["gpu_support_num"] = df["Supported GPU Length"].apply(first_number)
                df["cooler_support_num"] = df["Supported CPU Cooler Height"].apply(first_number)
                df["window_flag"] = df["Window"].apply(parse_bool)
                df["dust_filter_flag"] = df["Dust Filter"].apply(parse_bool)
                df["cable_mgmt_flag"] = df["Cable Management"].apply(parse_bool)
                df["mb_support_norm"] = df["Motherboard"].astype(str).str.upper().str.strip()
                df["psu_support_norm"] = df["Power Supply"].astype(str).str.upper().str.strip()
            elif name == "cpucooler":
                df["height_num"] = df["Height"].apply(first_number)
                df["tdp_num"] = df["TDP"].apply(first_number)
                df["supported_sockets_list"] = df["Supported Sockets"].apply(supported_socket_list)
            elif name == "monitor":
                df["pixels_num"] = df["Resolution"].apply(parse_resolution_pixels)
                df["refresh_num"] = df["Refresh Rate"].apply(first_number)
                df["size_num"] = df["Size"].apply(first_number)
                df["speaker_flag"] = df["Speaker"].apply(parse_bool)
                df["sync_flag"] = df["Sync"].astype(str).str.strip().ne("").astype(int)

            numeric_cols = df.select_dtypes(include=[np.number]).columns
            for col in numeric_cols:
                df[col] = df[col].fillna(df[col].median())

            self.data[name] = df

    # ---------- Stage 1 ----------
    def train_stage1(self):
        cpu = self.data["cpu"].copy()
        gpu = self.data["gpu"].copy()

        cpu["perf_target"] = (
            cpu["cores_num"] * 0.9
            + cpu["threads_num"] * 0.35
            + cpu["base_clock_num"] * 6.0
            + cpu["turbo_clock_num"] * 4.0
            + cpu["igpu_flag"] * 0.5
            - cpu["tdp_num"] * 0.01
        )
        gpu["perf_target"] = (
            gpu["vram_num"] * 2.0
            + gpu["boost_clock_num"] * 0.01
            + gpu["memory_clock_num"] * 0.002
            - gpu["tdp_num"] * 0.015
            + gpu["pin8_num"] * 1.5
        )

        cpu_X = cpu[["cores_num", "threads_num", "base_clock_num", "turbo_clock_num", "tdp_num", "igpu_flag"]]
        gpu_X = gpu[["vram_num", "boost_clock_num", "memory_clock_num", "tdp_num", "pin8_num", "pin6_num", "length_num"]]

        cpu_model = LinearRegression().fit(cpu_X, cpu["perf_target"])
        gpu_model = LinearRegression().fit(gpu_X, gpu["perf_target"])

        cpu["pred_perf"] = cpu_model.predict(cpu_X)
        gpu["pred_perf"] = gpu_model.predict(gpu_X)

        cpu["utility_per_dollar"] = cpu["pred_perf"] / cpu["price_num"].clip(lower=1.0)
        gpu["utility_per_dollar"] = gpu["pred_perf"] / gpu["price_num"].clip(lower=1.0)

        self.data["cpu"] = cpu
        self.data["gpu"] = gpu
        self.stage1_models = {"cpu": cpu_model, "gpu": gpu_model}

    def estimate_base_cost(self, goal: str = "balanced"):
        ram = self.data["ram"]
        ssd = self.data["ssd"]
        hdd = self.data["hdd"]
        mb = self.data["motherboard"]
        psu = self.data["psu"]
        case = self.data["case"]
        cooler = self.data["cpucooler"]
        monitor = self.data["monitor"]

        # conservative-but-affordable mandatory reserve
        reserve = (
            mb["price_num"].quantile(0.35)
            + ram["price_num"].quantile(0.35)
            + ssd["price_num"].quantile(0.35)
            + hdd["price_num"].quantile(0.20)
            + psu["price_num"].quantile(0.35)
            + case["price_num"].quantile(0.35)
            + cooler["price_num"].quantile(0.35)
            + monitor["price_num"].quantile(0.25)
        )
        if goal.lower() == "speed":
            reserve += 40  # leave room for better RAM/SSD
        return float(reserve)

    def allocate_budget(self, total_budget: float, goal: str = "balanced"):
        base_cost = self.estimate_base_cost(goal)
        flex_budget = max(total_budget - base_cost, total_budget * 0.35)

        cpu = self.data["cpu"]
        gpu = self.data["gpu"]

        if goal.lower() in {"gaming", "performance"}:
            gpu_share = 0.58
            cpu_share = 0.42
        elif goal.lower() == "speed":
            gpu_share = 0.48
            cpu_share = 0.52
        else:
            gpu_share = 0.52
            cpu_share = 0.48

        target_cpu_budget = flex_budget * cpu_share
        target_gpu_budget = flex_budget * gpu_share

        cpu_candidates = cpu[cpu["price_num"] <= max(target_cpu_budget * 1.2, cpu["price_num"].min() + 1)].copy()
        gpu_candidates = gpu[gpu["price_num"] <= max(target_gpu_budget * 1.2, gpu["price_num"].min() + 1)].copy()

        if cpu_candidates.empty:
            cpu_candidates = cpu.nsmallest(20, "price_num").copy()
        if gpu_candidates.empty:
            gpu_candidates = gpu.nsmallest(20, "price_num").copy()

        cpu_candidates["fit_score"] = cpu_candidates["utility_per_dollar"] - abs(cpu_candidates["price_num"] - target_cpu_budget) / max(target_cpu_budget, 1)
        gpu_candidates["fit_score"] = gpu_candidates["utility_per_dollar"] - abs(gpu_candidates["price_num"] - target_gpu_budget) / max(target_gpu_budget, 1)

        best_cpu = cpu_candidates.sort_values(["fit_score", "pred_perf"], ascending=False).iloc[0]
        best_gpu = gpu_candidates.sort_values(["fit_score", "pred_perf"], ascending=False).iloc[0]

        return {
            "base_cost": base_cost,
            "flex_budget": flex_budget,
            "cpu_budget": target_cpu_budget,
            "gpu_budget": target_gpu_budget,
            "anchor_cpu": best_cpu,
            "anchor_gpu": best_gpu,
        }

    # ---------- Stage 3 ----------
    def _compatibility_rule_features(self, combo: Dict[str, pd.Series]) -> Dict[str, float]:
        cpu, gpu = combo["cpu"], combo["gpu"]
        mb, ram = combo["motherboard"], combo["ram"]
        ssd, hdd = combo["ssd"], combo["hdd"]
        psu, case = combo["psu"], combo["case"]
        cooler, monitor = combo["cpucooler"], combo["monitor"]

        socket_match = int(cpu["socket_norm"] == mb["socket_norm"])
        cooler_socket_match = int(cpu["socket_norm"] in cooler["supported_sockets_list"])
        ram_match = int(ram["memory_type_norm"] == mb["memory_type_norm"])
        gpu_fit_margin = float(case["gpu_support_num"] - gpu["length_num"])
        cooler_fit_margin = float(case["cooler_support_num"] - cooler["height_num"])
        mb_fit = int(motherboard_form_factor(mb["form_factor_norm"]) in case_supported_form_factors(case["mb_support_norm"]))
        psu_fit = int(str(psu["size_norm"]).upper() == str(case["psu_support_norm"]).upper())

        total_tdp = float(cpu["tdp_num"] + gpu["tdp_num"] + 30 + ram["sticks_num"] * 3 + ssd["size_num"] * 0.005 + hdd["size_num"] * 0.004)
        required_watt = total_tdp * 1.25
        power_margin = float(psu["watt_num"] - required_watt)
        cooler_tdp_margin = float(cooler["tdp_num"] - cpu["tdp_num"])

        return {
            "socket_match": socket_match,
            "cooler_socket_match": cooler_socket_match,
            "ram_match": ram_match,
            "gpu_fit_margin": gpu_fit_margin,
            "cooler_fit_margin": cooler_fit_margin,
            "power_margin": power_margin,
            "mb_fit": mb_fit,
            "psu_fit": psu_fit,
            "cooler_tdp_margin": cooler_tdp_margin,
            "total_tdp": total_tdp,
            "monitor_refresh": float(monitor["refresh_num"]),
            "ssd_size": float(ssd["size_num"]),
            "hdd_size": float(hdd["size_num"]),
        }

    def train_stage3(self, n_samples: int = 2500, random_state: int = 42):
        rng = np.random.default_rng(random_state)
        rows = []

        categories = list(self.data.keys())
        for _ in range(n_samples):
            combo = {cat: self.data[cat].iloc[rng.integers(0, len(self.data[cat]))] for cat in categories}
            feats = self._compatibility_rule_features(combo)

            label = int(
                feats["socket_match"] == 1
                and feats["cooler_socket_match"] == 1
                and feats["ram_match"] == 1
                and feats["gpu_fit_margin"] >= 0
                and feats["cooler_fit_margin"] >= 0
                and feats["power_margin"] >= 0
                and feats["mb_fit"] == 1
                and feats["psu_fit"] == 1
                and feats["cooler_tdp_margin"] >= 0
            )
            rows.append({**feats, "label": label})

        train_df = pd.DataFrame(rows)
        X = train_df.drop(columns="label")
        y = train_df["label"]

        model = LogisticRegression(max_iter=1500, class_weight="balanced", solver="liblinear")
        model.fit(X, y)
        self.stage3_model = model
        self.stage3_feature_columns = X.columns.tolist()

    # ---------- Stage 2 ----------
    def _component_quality(self, row: pd.Series, category: str, goal: str) -> float:
        goal = goal.lower()
        if category == "cpu":
            return float(row.get("pred_perf", 0.0))
        if category == "gpu":
            score = float(row.get("pred_perf", 0.0))
            if goal in {"gaming", "performance"}:
                score *= 1.15
            return score
        if category == "ram":
            score = float(row["clock_num"]) + 0.6 * float(row["size_num"])
            if goal == "speed":
                score *= 1.20
            return score
        if category == "ssd":
            score = float(row["size_num"]) + 350 * float(row["nvme_flag"]) + 100 * float(row["tlc_flag"])
            if goal == "speed":
                score *= 1.25
            return score
        if category == "hdd":
            return float(row["size_num"]) + 0.1 * float(row["rpm_num"])
        if category == "motherboard":
            return float(row["mem_capacity_num"]) + 15 * float(row["ram_slots_num"]) + 25 * float(row["wifi_flag"])
        if category == "psu":
            return float(row["watt_num"]) + 20 * float(row["efficiency_score"])
        if category == "case":
            return float(row["gpu_support_num"]) + 0.5 * float(row["cooler_support_num"]) + 15 * float(row["dust_filter_flag"]) + 10 * float(row["cable_mgmt_flag"])
        if category == "cpucooler":
            return float(row["tdp_num"]) + 0.3 * float(row["height_num"])
        if category == "monitor":
            score = float(row["pixels_num"]) / 1e5 + float(row["refresh_num"]) + 2 * float(row["sync_flag"])
            if goal in {"gaming", "performance"}:
                score *= 1.10
            elif goal == "speed":
                score *= 1.02
            return score
        return 0.0

    def _candidate_pool(self, budget_plan: Dict, total_budget: float, goal: str) -> Dict[str, pd.DataFrame]:
        pools = {}
        for cat, df in self.data.items():
            temp = df.copy()
            temp["quality"] = temp.apply(lambda r: self._component_quality(r, cat, goal), axis=1)
            temp["value"] = temp["quality"] / temp["price_num"].clip(lower=1.0)

            if cat == "cpu":
                cap = budget_plan["cpu_budget"] * 1.30
                temp = temp[temp["price_num"] <= max(cap, temp["price_num"].quantile(0.4))]
            elif cat == "gpu":
                cap = budget_plan["gpu_budget"] * 1.30
                temp = temp[temp["price_num"] <= max(cap, temp["price_num"].quantile(0.4))]
            else:
                cap = total_budget * 0.25
                temp = temp[temp["price_num"] <= max(cap, temp["price_num"].quantile(0.5))]

            if temp.empty:
                temp = df.copy()
                temp["quality"] = temp.apply(lambda r: self._component_quality(r, cat, goal), axis=1)
                temp["value"] = temp["quality"] / temp["price_num"].clip(lower=1.0)

            pools[cat] = temp.sort_values(["value", "quality"], ascending=False).head(30).reset_index(drop=True)
        return pools

    def train_stage2(self, budget_plan: Dict, total_budget: float, goal: str = "balanced", n_samples: int = 1200, random_state: int = 42):
        rng = np.random.default_rng(random_state)
        pools = self._candidate_pool(budget_plan, total_budget, goal)

        rows = []
        categories = list(pools.keys())
        for _ in range(n_samples):
            combo = {cat: pools[cat].iloc[rng.integers(0, len(pools[cat]))] for cat in categories}
            comp_feats = self._compatibility_rule_features(combo)
            compat_penalty = (
                25 * comp_feats["socket_match"]
                + 20 * comp_feats["cooler_socket_match"]
                + 20 * comp_feats["ram_match"]
                + 10 * comp_feats["mb_fit"]
                + 10 * comp_feats["psu_fit"]
                + min(comp_feats["gpu_fit_margin"], 50) * 0.2
                + min(comp_feats["cooler_fit_margin"], 40) * 0.25
                + min(comp_feats["power_margin"], 150) * 0.08
            )

            total_cost = sum(float(combo[cat]["price_num"]) for cat in categories)
            over_budget_penalty = max(0.0, total_cost - total_budget) * 0.55

            quality_sum = 0.0
            for cat in categories:
                quality_sum += self._component_quality(combo[cat], cat, goal)

            # synthetic training target for ranking
            target = quality_sum + compat_penalty - over_budget_penalty
            if goal.lower() == "speed":
                target += combo["ram"]["clock_num"] * 0.35 + combo["ssd"]["nvme_flag"] * 120 + combo["ssd"]["size_num"] * 0.03
            elif goal.lower() in {"gaming", "performance"}:
                target += combo["gpu"]["pred_perf"] * 0.8 + combo["monitor"]["refresh_num"] * 0.5

            rows.append({
                "cpu_perf": combo["cpu"]["pred_perf"],
                "gpu_perf": combo["gpu"]["pred_perf"],
                "mb_memcap": combo["motherboard"]["mem_capacity_num"],
                "ram_clock": combo["ram"]["clock_num"],
                "ram_size": combo["ram"]["size_num"],
                "ssd_size": combo["ssd"]["size_num"],
                "ssd_nvme": combo["ssd"]["nvme_flag"],
                "hdd_size": combo["hdd"]["size_num"],
                "psu_watt": combo["psu"]["watt_num"],
                "psu_eff": combo["psu"]["efficiency_score"],
                "case_gpu_support": combo["case"]["gpu_support_num"],
                "cooler_tdp": combo["cpucooler"]["tdp_num"],
                "monitor_pixels": combo["monitor"]["pixels_num"],
                "monitor_refresh": combo["monitor"]["refresh_num"],
                "price_total": total_cost,
                "compat_rule_score": compat_penalty,
                "target": target,
            })

        train_df = pd.DataFrame(rows)
        X = train_df.drop(columns="target")
        y = train_df["target"]

        model = RandomForestRegressor(
            n_estimators=120,
            max_depth=14,
            min_samples_leaf=3,
            random_state=random_state,
            n_jobs=1,
        )
        model.fit(X, y)
        self.stage2_model = model
        self.stage2_feature_columns = X.columns.tolist()
        self.stage2_pools = pools

    def _build_stage2_features(self, combo: Dict[str, pd.Series]) -> pd.DataFrame:
        comp_feats = self._compatibility_rule_features(combo)
        row = pd.DataFrame([{
            "cpu_perf": combo["cpu"]["pred_perf"],
            "gpu_perf": combo["gpu"]["pred_perf"],
            "mb_memcap": combo["motherboard"]["mem_capacity_num"],
            "ram_clock": combo["ram"]["clock_num"],
            "ram_size": combo["ram"]["size_num"],
            "ssd_size": combo["ssd"]["size_num"],
            "ssd_nvme": combo["ssd"]["nvme_flag"],
            "hdd_size": combo["hdd"]["size_num"],
            "psu_watt": combo["psu"]["watt_num"],
            "psu_eff": combo["psu"]["efficiency_score"],
            "case_gpu_support": combo["case"]["gpu_support_num"],
            "cooler_tdp": combo["cpucooler"]["tdp_num"],
            "monitor_pixels": combo["monitor"]["pixels_num"],
            "monitor_refresh": combo["monitor"]["refresh_num"],
            "price_total": sum(float(combo[k]["price_num"]) for k in combo),
            "compat_rule_score": (
                25 * comp_feats["socket_match"]
                + 20 * comp_feats["cooler_socket_match"]
                + 20 * comp_feats["ram_match"]
                + 10 * comp_feats["mb_fit"]
                + 10 * comp_feats["psu_fit"]
            ),
        }])
        return row[self.stage2_feature_columns]

    def _compatibility_probability(self, combo: Dict[str, pd.Series]) -> float:
        feats = self._compatibility_rule_features(combo)
        X = pd.DataFrame([[feats[c] for c in self.stage3_feature_columns]], columns=self.stage3_feature_columns)
        return float(self.stage3_model.predict_proba(X)[0, 1])

    def generate_top_builds(self, total_budget: float, goal: str = "balanced", top_k: int = 3, n_trials: int = 2000, random_state: int = 42) -> List[Build]:
        self.train_stage1()
        budget_plan = self.allocate_budget(total_budget, goal)
        self.train_stage3()
        self.train_stage2(budget_plan, total_budget, goal)

        rng = np.random.default_rng(random_state)
        builds = []
        pools = self.stage2_pools
        categories = list(pools.keys())

        for _ in range(n_trials):
            combo = {cat: pools[cat].iloc[rng.integers(0, len(pools[cat]))] for cat in categories}
            total_cost = sum(float(combo[cat]["price_num"]) for cat in categories)
            if total_cost > total_budget:
                continue

            compat_prob = self._compatibility_probability(combo)
            if compat_prob < 0.55:
                continue

            X = self._build_stage2_features(combo)
            score = float(self.stage2_model.predict(X)[0])

            # final balanced score: ML rank + gentle boost for compatibility + budget efficiency
            budget_left = total_budget - total_cost
            final_score = score + compat_prob * 80 - abs(budget_left) * 0.05
            builds.append(Build(parts=combo, total_cost=total_cost, compatibility_probability=compat_prob, score=final_score))

        # fallback: if too few builds, relax threshold a bit
        if len(builds) < top_k:
            for _ in range(n_trials // 2):
                combo = {cat: pools[cat].iloc[rng.integers(0, len(pools[cat]))] for cat in categories}
                total_cost = sum(float(combo[cat]["price_num"]) for cat in categories)
                if total_cost > total_budget:
                    continue
                compat_prob = self._compatibility_probability(combo)
                X = self._build_stage2_features(combo)
                score = float(self.stage2_model.predict(X)[0])
                final_score = score + compat_prob * 80 - abs(total_budget - total_cost) * 0.05
                builds.append(Build(parts=combo, total_cost=total_cost, compatibility_probability=compat_prob, score=final_score))

        # de-duplicate by name tuple
        unique = {}
        for b in sorted(builds, key=lambda x: x.score, reverse=True):
            key = tuple(b.parts[cat]["Name"] for cat in categories)
            if key not in unique:
                unique[key] = b

        ranked = list(unique.values())
        ranked.sort(key=lambda x: x.score, reverse=True)
        return ranked[:top_k]


In [25]:
def build_summary(build: Build) -> Dict:
    return {
        "total_cost": round(build.total_cost, 2),
        "compatibility_probability": round(build.compatibility_probability, 4),
        "score": round(build.score, 2),
        "parts": {
            cat: {
                "name": str(row["Name"]),
                "price": round(float(row["price_num"]), 2),
                "url": row.get("Product Page", None),
            }
            for cat, row in build.parts.items()
        },
    }

if __name__ == "__main__":
    optimizer = PCBuildOptimizer(".")
    builds = optimizer.generate_top_builds(total_budget=1200, goal="speed", top_k=3, n_trials=1200)
    for i, build in enumerate(builds, 1):
        print(f"Build #{i}")
        summary = build_summary(build)
        print(f"Total Cost: ${summary['total_cost']}")
        print(f"Compatibility Probability: {summary['compatibility_probability']}")
        for cat, part in summary["parts"].items():
            print(f"  - {cat:12s}: {part['name']} (${part['price']})")
        print("-" * 70)


Build #1
Total Cost: $968.72
Compatibility Probability: 0.9833
  - cpu         : AMD Ryzen 3 3100 ($120.4)
  - gpu         : Asus PH RX 550 4G M7 ($138.94)
  - motherboard : Gigabyte GA-AB350M-DS3H ($54.4)
  - ram         : Crucial Ballistix Sport LT series white ($37.57)
  - ssd         : Crucial BX500 - 2 TB 2000 GB ($96.67)
  - hdd         : Western Digital Red Pro 14 TB ($297.38)
  - psu         : Thermaltake Smart Pro RGB 80Plus Bronze modular ($72.53)
  - case        : be quiet! Pure Base 600 Midi-Tower - orange Window ($24.95)
  - cpucooler   : Arctic Alpine 64 plus - 92mm ($17.08)
  - monitor     : ASUSTUF Gaming VG289Q1A ($108.8)
----------------------------------------------------------------------
Build #2
Total Cost: $981.16
Compatibility Probability: 0.9504
  - cpu         : AMD Ryzen 3 2200G ($101.55)
  - gpu         : Asus AREZ PH RX 550 2G ($112.43)
  - motherboard : MSI B550M Pro-VDH WiFi ($123.31)
  - ram         : Ballistix Tactical DDR4-3000 CL15 ($28.35)
  - ssd   

### Step 1: Initialize Git and clone your repository (if it's not already cloned)

In [26]:
# Replace 'YOUR_GITHUB_USERNAME' and 'YOUR_REPOSITORY_NAME' with your actual details.
# If you haven't initialized a repository, you might want to create one on GitHub first.
# Then clone it here.

# Example: Cloning an existing repository
# !git clone https://github.com/YOUR_GITHUB_USERNAME/YOUR_REPOSITORY_NAME.git

# If you are starting from scratch in Colab and want to push an existing notebook
# (e.g., this current notebook), you'll first create a new empty repository on GitHub,
# then initialize git locally, add your files, commit, add the remote, and push.

# Go to the content directory where your notebook is usually saved
%cd /content

# Initialize a new Git repository in the current directory
!git init

# Add the notebook file to the repository. Replace 'Your_Notebook_Name.ipynb' with your notebook's name.
# To find your notebook's name, you can check the 'Files' tab on the left sidebar.
# Or, if you just ran a cell, it might be the default name 'Untitled.ipynb' or similar.
# For example, if your notebook is 'MyProject.ipynb':
# !git add MyProject.ipynb

# If you want to add all files in the current directory:
!git add .


/content
hint: Using 'master' as the name for the initial branch. This default branch name
hint: is subject to change. To configure the initial branch name to use in all
hint: of your new repositories, which will suppress this warning, call:
hint: 
hint: 	git config --global init.defaultBranch <name>
hint: 
hint: Names commonly chosen instead of 'master' are 'main', 'trunk' and
hint: 'development'. The just-created branch can be renamed via this command:
hint: 
hint: 	git branch -m <name>
Initialized empty Git repository in /content/.git/


### Step 2: Configure Git and commit your changes

In [27]:
# Set your user name and email (required for Git commits)
!git config --global user.email "you@example.com"
!git config --global user.name "Your Name"

# Commit the changes
!git commit -m "Initial commit of Colab notebook"


[master (root-commit) feefb0a] Initial commit of Colab notebook
 31 files changed, 54033 insertions(+)
 create mode 100644 .config/.last_opt_in_prompt.yaml
 create mode 100644 .config/.last_survey_prompt.yaml
 create mode 100644 .config/.last_update_check.json
 create mode 100644 .config/active_config
 create mode 100644 .config/config_sentinel
 create mode 100644 .config/configurations/config_default
 create mode 100644 .config/default_configs.db
 create mode 100644 .config/gce
 create mode 100644 .config/hidden_gcloud_config_universe_descriptor_data_cache_configs.db
 create mode 100644 .config/logs/2026.04.02/13.30.17.544197.log
 create mode 100644 .config/logs/2026.04.02/13.30.40.372331.log
 create mode 100644 .config/logs/2026.04.02/13.30.51.422062.log
 create mode 100644 .config/logs/2026.04.02/13.30.52.812826.log
 create mode 100644 .config/logs/2026.04.02/13.31.06.236912.log
 create mode 100644 .config/logs/2026.04.02/13.31.07.077226.log
 create mode 100644 CPUCoolerData.csv
 cr

### Step 3: Add the remote repository and push

In [28]:
# Add your remote repository URL. Replace 'YOUR_GITHUB_USERNAME' and 'YOUR_REPOSITORY_NAME'.
# This should be the HTTPS URL of the repository you created on GitHub.
# Example: https://github.com/YOUR_GITHUB_USERNAME/YOUR_REPOSITORY_NAME.git
!git remote add origin https://github.com/YOUR_GITHUB_USERNAME/YOUR_REPOSITORY_NAME.git

# If you previously added 'origin' and need to change it, use:
# !git remote set-url origin https://github.com/YOUR_GITHUB_USERNAME/YOUR_REPOSITORY_NAME.git

# Rename the local 'master' branch to 'main' to align with GitHub's default
!git branch -M main

# Push your changes to the main branch
# Git will prompt you for your GitHub username and Personal Access Token (PAT).
# You should create a PAT on GitHub under Developer Settings > Personal access tokens.
!git push -u origin main


error: src refspec main does not match any
error: failed to push some refs to 'https://github.com/YOUR_GITHUB_USERNAME/YOUR_REPOSITORY_NAME.git'


After running the `git push` command, a dialog box will appear asking for your GitHub username and password/Personal Access Token (PAT). Enter these securely to complete the push operation.